# scPhyloGRN minimal training demo

This notebook generates deterministic synthetic inputs, runs the repository's actual training CLI for two epochs, and inspects its ranked edge output. It is a functional smoke test and interface tutorial, not a biological benchmark.

In [ ]:
from pathlib import Path
import subprocess
import sys
import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'run_scPhyloGRN.py').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the scPhyloGRN repository.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
WORK = REPO_ROOT / 'examples' / 'training_demo' / 'work'
WORK.mkdir(parents=True, exist_ok=True)
REPO_ROOT, WORK

In [ ]:
rng = np.random.default_rng(42)
n_genes, n_cells = 30, 24
genes = [f'Gene{i:02d}' for i in range(n_genes)]
cells = [f'Cell{i:02d}' for i in range(n_cells)]
time = np.linspace(-1.0, 1.0, n_cells)
expression = rng.normal(0, 0.35, size=(n_genes, n_cells))
for i in range(n_genes):
    expression[i] += np.sin((i % 5 + 1) * time) + (i % 3 - 1) * time
expr_path = WORK / 'synthetic_expression.csv'
pd.DataFrame(expression, index=genes, columns=cells).to_csv(expr_path)

ref_path = WORK / 'synthetic_reference_edges.csv'
pd.DataFrame({'gene_i': genes, 'gene_j': genes[1:] + genes[:1]}).to_csv(ref_path, index=False)

distance = np.abs(time[:, None] - time[None, :])
kernel = np.exp(-distance / 0.35).astype(np.float32)
kernel_path = WORK / 'synthetic_lineage_K.npy'
np.save(kernel_path, kernel)
print(f'Expression: {expression.shape}; reference edges: {n_genes}; kernel: {kernel.shape}')

In [ ]:
output_dir = WORK / 'results'
command = [
    sys.executable, str(REPO_ROOT / 'src' / 'run_scPhyloGRN.py'),
    '--expr', str(expr_path),
    '--ref', str(ref_path),
    '--lineage-k', str(kernel_path),
    '--outdir', str(output_dir),
    '--epochs', '2',
    '--batch_size', '64',
    '--proj_dim', '16',
    '--out_dim', '16',
    '--heads', '2',
    '--depth', '1',
    '--k_top', '29',
    '--smooth_scales', '1',
    '--dropout', '0.1',
    '--seed', '42',
    '--no_calibration',
]
print(' '.join(command))
subprocess.run(command, cwd=REPO_ROOT, check=True)

In [ ]:
prediction_path = max(output_dir.glob('predicted_edges_*.csv'), key=lambda p: p.stat().st_mtime)
predictions = pd.read_csv(prediction_path).sort_values('prob_T', ascending=False)
print(f'Loaded {len(predictions):,} candidate edges from {prediction_path.name}')
predictions.head(10)